# Tracing an Ejentum cognitive harness agent loop in Phoenix

This notebook shows how to trace an agent that calls the [Ejentum cognitive harness](https://ejentum.com) REST API between an LLM call and its response, so each turn shows up in Phoenix with the harness retrieval, the model call, and their token usage as nested spans.

The pattern generalises to any third-party tool the agent calls in-loop. Ejentum is the worked example because the agent calls a single REST endpoint with a `mode` argument, which keeps the OpenTelemetry side small.

## What you'll set up

1. A local Phoenix session for collecting traces.
2. Auto-instrumented OpenAI calls via `openinference-instrumentation-openai`.
3. A small manual span around each Ejentum REST call so the harness retrieval is visible alongside the model call.
4. A two-turn agent that calls `harness_reasoning` before answering a reasoning-heavy task.

Open the Phoenix UI at the URL printed below to inspect the trace tree per turn.

## 1. Install

In [ ]:
%pip install -q "arize-phoenix>=4.0.0" openai openinference-instrumentation-openai opentelemetry-api opentelemetry-sdk requests

## 2. Launch Phoenix and instrument OpenAI

In [ ]:
import os

import phoenix as px
from openinference.instrumentation.openai import OpenAIInstrumentor
from phoenix.otel import register

# Phoenix collector session. Open this URL in your browser to view traces.
session = px.launch_app()
print(f"Phoenix UI: {session.url}")

tracer_provider = register(project_name="ejentum-harness-demo")
OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
tracer = tracer_provider.get_tracer(__name__)

## 3. Wrap Ejentum REST calls in a manual span

OpenAI calls are auto-instrumented above. For Ejentum we want a thin manual span so the harness retrieval shows up in the trace tree alongside the model call, with the `mode` and a snippet of the returned scaffold attached as span attributes.

Get an Ejentum key at <https://ejentum.com/dashboard>. Free and paid tiers are available.

In [ ]:
import requests
from opentelemetry.trace import Status, StatusCode

EJENTUM_URL = "https://ejentum-main-ab125c3.zuplo.app/logicv1/"


def call_ejentum(query: str, mode: str = "reasoning") -> str:
    """Fetch a cognitive scaffold from the Ejentum REST gateway.

    Emits a single OTel span with attributes: ejentum.mode, ejentum.query.length,
    ejentum.scaffold.length. The returned scaffold string is what the model
    reads before answering.
    """
    with tracer.start_as_current_span("ejentum.call") as span:
        span.set_attribute("ejentum.mode", mode)
        span.set_attribute("ejentum.query.length", len(query))
        try:
            r = requests.post(
                EJENTUM_URL,
                headers={
                    "Authorization": f"Bearer {os.environ['EJENTUM_API_KEY']}",
                    "Content-Type": "application/json",
                },
                json={"query": query, "mode": mode},
                timeout=10,
            )
            r.raise_for_status()
            payload = r.json()
            scaffold = payload[0].get(mode, "") if isinstance(payload, list) else ""
            span.set_attribute("ejentum.scaffold.length", len(scaffold))
            return scaffold
        except Exception as err:
            span.set_status(Status(StatusCode.ERROR, str(err)))
            raise

## 4. Run a two-turn agent and view the traces

Each turn does the same shape:

1. Call `call_ejentum` (one OTel span).
2. Call `openai.chat.completions.create` with the scaffold spliced into the system message (auto-instrumented spans for the OpenAI call and its tokens).

Open the Phoenix URL printed earlier to inspect the trace tree.

In [ ]:
from openai import OpenAI

openai_client = OpenAI()


def harness_augmented_turn(task: str, mode: str = "reasoning") -> str:
    scaffold = call_ejentum(task, mode=mode)
    completion = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": (
                    "Apply the cognitive scaffold below before answering.\n\n"
                    f"[SCAFFOLD]\n{scaffold}\n[END]"
                ),
            },
            {"role": "user", "content": task},
        ],
        temperature=0,
    )
    return completion.choices[0].message.content or ""


print(harness_augmented_turn(
    "We have 50M users and want to add a NOT NULL column. "
    "Walk through the trade-offs and recommend a migration path.",
    mode="reasoning",
))
print("---")
print(harness_augmented_turn(
    "A user insists their DB is the bottleneck and refuses other diagnostics. "
    "Walk through whether the framing is sound before recommending.",
    mode="anti-deception",
))

## What to look at in Phoenix

Open the URL printed by `px.launch_app()`. For each turn you should see:

- A root span for the turn (the outer Python call).
- A child `ejentum.call` span with `ejentum.mode`, `ejentum.query.length`, and `ejentum.scaffold.length` attributes.
- A sibling `chat.completions.create` span (from the OpenAI auto-instrumentation) with token-usage details on its attributes.

Use the **Traces** view to filter by `ejentum.mode = "anti-deception"` to see only deception-resistance turns, or by `ejentum.scaffold.length > 0` to confirm every harness call returned a non-empty scaffold.

## Adapting the pattern

Any in-loop REST call to a third-party tool can be instrumented the same way. The four steps in this notebook (manual span + attributes + auto-instrumented client + scaffold-aware system message) generalise to harness calls in any agent loop, not just the two-turn shape shown here. For larger agent frameworks (CrewAI, LangChain, AutoGen), use the matching `openinference-instrumentation-<framework>` package alongside this manual span.